# Monitoring a deployed system for drift on clustered data

**`AsymptoticClusteredPPRM`** and its label-only counterpart **`AsymptoticClassicalRM`** watch a quality metric across successive batches of production data and raise an alarm the moment there is statistically valid evidence that it has drifted past a threshold you fix in advance. `AsymptoticClusteredPPRM` is built for data where samples are grouped into clusters (such as turns within conversations), because samples of the same cluster are correlated and must be treated as a single unit. This is necessary because a deployed system's quality is rarely static: traffic shifts and upstream models change, so a metric that looked fine at launch can drift silently, and a single-shot estimate made at first deployment time cannot catch that.

---

**What you will learn:**

- Why re-checking a fresh confidence interval multiple times inflates the false-alarm rate
- Why ignoring the cluster structure produces overconfident per-batch intervals
- How `AsymptoticClusteredPPRM` and `AsymptoticClassicalRM` give an anytime-valid alternative that stays safe under repeated checks
- How to read their running-mean curves, confidence bounds, and alarms

## The problem: the metric can drift after launch, and ignoring conversations breaks validity

Suppose you have a customer-facing AI assistant that handles thousands of multi-turn conversations, where hallucinations are a real risk. Every week, a fresh batch of conversations is judged turn by turn by your LLM judge, which is systematically optimistic, and a handful of whole conversations are sent for human review. Turns of the same conversation share context, so they are correlated and must be treated as a single unit, a **cluster**. Your team has fixed a business threshold on the hallucination rate, and wants to know as soon as it has drifted past it since launch.

The tempting approach is to compute a fresh confidence interval every week and raise an alarm the moment one crosses the threshold. This is **peeking**: each weekly test carries its own false-alarm probability, and checking after every batch compounds those chances, so a false alarm becomes likely over a long enough monitoring horizon even if nothing has actually drifted. The [Monitors user guide](../../../user_guide/monitors/) explains why this happens and how an anytime-valid confidence sequence that stays safe no matter how often you look avoids it.

A second trap is specific to this setting: treating every turn as an independent observation. Turns of the same conversation are correlated, so a batch carries less independent information than its number of turns suggests, and intervals that ignore this are too narrow, which makes false alarms even more likely. This tutorial focuses on what that means for you in practice.

In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np

from glide.estimators import ClusteredPPIMeanEstimator, PPIMeanEstimator
from glide.monitors import AsymptoticClassicalRM, AsymptoticClusteredPPRM
from glide.samplers import UniformClusteredSampler
from glide.simulators import generate_batched_clustered_binary_dataset, simulate_annotation

# Colour palette
C_ESTIMATE = "#95A5A6"  # cluster-aware per-batch estimate: grey
C_ESTIMATE_ALARM = "#566573"  # cluster-aware naive baseline alarm: dark grey
C_NAIVE = "#E74C3C"  # cluster-blind naive baseline: red-orange
C_PPRM = "#27AE60"  # clustered PPRM running mean/bound: green
C_CLASSICAL = "#2980B9"  # classical monitor running mean/bound: blue
C_ALARM_PPRM = "#8E44AD"  # clustered PPRM alarm: purple
C_ALARM_CLASSICAL = "#F39C12"  # classical monitor alarm: gold
C_THRESHOLD = "#2C3E50"  # threshold and true rate: dark slate

# Global plot style
plt.rcParams.update(
    {
        "figure.facecolor": "white",
        "axes.facecolor": "#FAFAFA",
        "axes.grid": True,
        "grid.color": "#E5E5E5",
        "grid.linewidth": 0.8,
        "font.size": 18,
        "axes.labelsize": 18,
        "axes.titlesize": 18,
        "legend.fontsize": 14,
        "xtick.labelsize": 16,
        "ytick.labelsize": 16,
        "figure.titlesize": 19,
    }
)

## Simulating a batched, clustered production stream

We use `generate_batched_clustered_binary_dataset` to simulate successive batches of conversations, each with its own true hallucination rate, stacked into a single stream. Every row is a conversation turn and carries two identifiers: an integer for the chronological batch it belongs to, and an integer for the conversation (cluster) it belongs to. Conversation identifiers are unique across the whole stream, since a conversation never spans two weeks.

`simulate_batches` below builds such a stream from a list of per-week true hallucination rates. Every week gets `BATCH_SIZE` judged turns spread over `N_CLUSTERS_PER_BATCH` conversations, of which `LABELED_CLUSTERS_PER_BATCH` conversations are sent for full human review. Because annotators review whole conversations, we use the `UniformClusteredSampler` to draw the review budget in terms of conversations rather than turns. We apply it once per week: a conversation only exists within its own week, so the review budget is drawn separately in each. The weekly selection indicators are then combined, and `simulate_annotation` reveals the human labels of the selected conversations only, leaving the rest as `np.nan`. The judge under-reports hallucinations by a constant bias. We fix an alarm threshold above the stationary rate and assess the monitoring methods' ability to detect when it is exceeded while avoiding false alarms. All batches in the stream are monitored.

In [ ]:
N_BATCHES = 50
BATCH_SIZE = 800
N_CLUSTERS_PER_BATCH = 80
LABELED_CLUSTERS_PER_BATCH = 15
STATIONARY_MEAN = 0.28
PROXY_BIAS = -0.03
CORRELATION = 0.9
CONFIDENCE_LEVEL = 0.8
THRESHOLD = 0.3


def simulate_batches(true_means, random_seed):
    n_batches = len(true_means)
    y_true_oracle, y_proxy, batches, clusters = generate_batched_clustered_binary_dataset(
        n_samples=[BATCH_SIZE] * n_batches,
        n_clusters=[N_CLUSTERS_PER_BATCH] * n_batches,
        true_mean=true_means,
        proxy_mean=true_means + PROXY_BIAS,
        correlation=[CORRELATION] * n_batches,
        random_seed=random_seed,
    )
    xi = np.zeros(len(y_true_oracle))
    for batch_id in range(n_batches):
        batch_mask = batches == batch_id
        xi[batch_mask] = UniformClusteredSampler().sample(
            clusters[batch_mask], n_clusters=LABELED_CLUSTERS_PER_BATCH, random_seed=random_seed + batch_id
        )
    y_true = simulate_annotation(y_true_oracle, xi)
    return y_true, y_proxy, clusters, batches

## The naive baselines: one confidence interval per batch

The tempting-but-invalid approach simply calls an estimator once per week at the same confidence level as the monitors, comparing that week's confidence interval to the threshold and alarming whenever the interval lies above it. We consider two versions of it:

- **Cluster-aware**: `ClusteredPPIMeanEstimator`, which accounts for the correlation between turns of the same conversation.
- **Cluster-blind**: `PPIMeanEstimator` applied to the very same batch, which treats every turn as an independent observation.

Both use the same labels and the same proxies, and only differ in how they account for the conversation structure. Their per-batch intervals are compared in Scenario 1.

In [ ]:
def naive_per_batch_alarms(y_true, y_proxy, clusters, batches, threshold, n_batches, *, cluster_aware):
    lower_bounds = np.zeros(n_batches)
    upper_bounds = np.zeros(n_batches)
    alarms = np.zeros(n_batches, dtype=bool)
    for batch_id in range(n_batches):
        batch_mask = batches == batch_id
        if cluster_aware:
            batch_result = ClusteredPPIMeanEstimator().estimate(
                y_true[batch_mask], y_proxy[batch_mask], clusters[batch_mask], confidence_level=CONFIDENCE_LEVEL
            )
        else:
            batch_result = PPIMeanEstimator().estimate(
                y_true[batch_mask], y_proxy[batch_mask], confidence_level=CONFIDENCE_LEVEL
            )
        lower_bounds[batch_id] = batch_result.confidence_interval.lower_bound
        upper_bounds[batch_id] = batch_result.confidence_interval.upper_bound
        alarms[batch_id] = batch_result.confidence_interval.lower_bound > threshold
    return lower_bounds, upper_bounds, alarms

In [ ]:
def plot_monitor_vs_baseline(
    lower_bounds, blind_lower_bounds, pprm_result, classical_result, naive_alarms, blind_naive_alarms, title
):
    weeks = np.arange(1, len(lower_bounds) + 1)
    fig, ax = plt.subplots(figsize=(10, 6.5))

    ax.scatter(
        weeks, lower_bounds, color=C_ESTIMATE, s=28, alpha=0.8, zorder=3, label="Per-batch lower bounds (cluster-aware)"
    )
    ax.scatter(
        weeks,
        blind_lower_bounds,
        color=C_NAIVE,
        s=28,
        alpha=0.5,
        zorder=3,
        label="Per-batch lower bounds (cluster-blind)",
    )
    ax.plot(
        weeks,
        pprm_result.confidence_bounds,
        color=C_PPRM,
        linewidth=2.5,
        zorder=4,
        label="Clustered PPRM lower bounds",
    )
    ax.plot(
        weeks,
        classical_result.confidence_bounds,
        color=C_CLASSICAL,
        linewidth=2.5,
        zorder=4,
        label="Classical monitor lower bounds",
    )
    ax.axhline(THRESHOLD, color=C_THRESHOLD, linestyle="--", linewidth=1.8, zorder=2, label="Threshold")

    if naive_alarms.any():
        ax.scatter(
            weeks[naive_alarms],
            lower_bounds[naive_alarms],
            marker="+",
            s=110,
            linewidths=2.5,
            color=C_ESTIMATE_ALARM,
            zorder=5,
            label="Naive alarms (cluster-aware)",
        )

    if blind_naive_alarms.any():
        ax.scatter(
            weeks[blind_naive_alarms],
            blind_lower_bounds[blind_naive_alarms],
            marker="+",
            s=110,
            linewidths=2.5,
            color=C_NAIVE,
            zorder=5,
            label="Naive alarms (cluster-blind)",
        )

    if pprm_result.drift_detected:
        alarm_week = pprm_result.first_alarm_index + 1
        ax.axvline(
            alarm_week,
            color=C_ALARM_PPRM,
            linestyle=":",
            linewidth=2.2,
            zorder=2,
            label=f"Clustered PPRM alarm (week {alarm_week})",
        )

    if classical_result.drift_detected:
        alarm_week = classical_result.first_alarm_index + 1
        ax.axvline(
            alarm_week,
            color=C_ALARM_CLASSICAL,
            linestyle=":",
            linewidth=2.2,
            zorder=2,
            label=f"Classical alarm (week {alarm_week})",
        )

    ax.set_xlabel("Week")
    ax.set_ylabel("Hallucination Rate")
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
    ax.set_title(title)
    ax.set_ylim(-0.05, 1.45)
    ax.set_yticks(np.arange(0, 1.01, 0.2))
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(loc="upper left", ncol=2, fontsize=11, framealpha=0.9)
    plt.tight_layout()
    plt.show()

## Scenario 1: no drift, no false alarm

We generate a fully stationary stream: the true hallucination rate stays below the threshold. Neither monitor should alarm, except by chance.

In [ ]:
true_means_no_drift = np.full(N_BATCHES, STATIONARY_MEAN)
y_true_no_drift, y_proxy_no_drift, clusters_no_drift, batches_no_drift = simulate_batches(
    true_means_no_drift, random_seed=8
)

n_labeled = np.sum(~np.isnan(y_true_no_drift))
print(f"Total turns            : {len(y_true_no_drift):,}")
print(f"  Human-reviewed turns : {n_labeled:,}")
print(f"Total conversations    : {len(np.unique(clusters_no_drift)):,}")
print(f"Number of weeks        : {len(np.unique(batches_no_drift))}")

In [ ]:
n_nan_in_y_true = np.isnan(y_true_no_drift).sum()
n_zeros_in_y_true = np.sum(y_true_no_drift == 0)
n_ones_in_y_true = np.sum(y_true_no_drift == 1)
print("Analysis of the human-reviewed labels:")
print(f"Number of NaNs: {n_nan_in_y_true}")
print(f"Number of 0s: {n_zeros_in_y_true}")
print(f"Number of 1s: {n_ones_in_y_true}")

In [ ]:
n_nan_in_y_proxy = np.isnan(y_proxy_no_drift).sum()
n_zeros_in_y_proxy = np.sum(y_proxy_no_drift == 0)
n_ones_in_y_proxy = np.sum(y_proxy_no_drift == 1)
print("Analysis of the proxy labels:")
print(f"Number of NaNs: {n_nan_in_y_proxy}")
print(f"Number of 0s: {n_zeros_in_y_proxy}")
print(f"Number of 1s: {n_ones_in_y_proxy}")

In [ ]:
turns_per_batch = np.bincount(batches_no_drift.astype(int))
conversations_per_batch = [len(np.unique(clusters_no_drift[batches_no_drift == i])) for i in range(N_BATCHES)]
print(
    f"There are {len(turns_per_batch)} batches, with {turns_per_batch[0]} turns "
    f"in {conversations_per_batch[0]} conversations per batch."
)

Run the naive baselines and both monitoring methods.

In [ ]:
lower_no_drift, upper_no_drift, naive_alarms_no_drift = naive_per_batch_alarms(
    y_true_no_drift,
    y_proxy_no_drift,
    clusters_no_drift,
    batches_no_drift,
    THRESHOLD,
    N_BATCHES,
    cluster_aware=True,
)
blind_lower_no_drift, blind_upper_no_drift, blind_naive_alarms_no_drift = naive_per_batch_alarms(
    y_true_no_drift,
    y_proxy_no_drift,
    clusters_no_drift,
    batches_no_drift,
    THRESHOLD,
    N_BATCHES,
    cluster_aware=False,
)
pprm_result_no_drift = AsymptoticClusteredPPRM().detect(
    y_true_no_drift,
    y_proxy_no_drift,
    clusters_no_drift,
    batches_no_drift,
    higher_is_better=False,
    threshold=THRESHOLD,
    confidence_level=CONFIDENCE_LEVEL,
    metric_name="Hallucination Rate",
)
classical_result_no_drift = AsymptoticClassicalRM().detect(
    y_true_no_drift,
    batches_no_drift,
    higher_is_better=False,
    threshold=THRESHOLD,
    confidence_level=CONFIDENCE_LEVEL,
    metric_name="Hallucination Rate",
)

print(f"Naive baseline alarms at week(s) (cluster-aware): {np.flatnonzero(naive_alarms_no_drift) + 1}")
print(f"Naive baseline alarms at week(s) (cluster-blind): {np.flatnonzero(blind_naive_alarms_no_drift) + 1}")
print(f"Mean per-batch interval width (cluster-aware): {np.mean(upper_no_drift - lower_no_drift):.1%}")
print(f"Mean per-batch interval width (cluster-blind): {np.mean(blind_upper_no_drift - blind_lower_no_drift):.1%}")
print(f"Clustered PPRM drift detected: {pprm_result_no_drift.drift_detected}")
print(f"Classical monitor drift detected: {classical_result_no_drift.drift_detected}")
print(
    f"Clustered PPRM running mean / bound after {N_BATCHES} weeks: {pprm_result_no_drift.running_means[-1]:.1%} / "
    f"{pprm_result_no_drift.confidence_bounds[-1]:.1%}"
)
print(
    f"Classical monitor running mean / bound after {N_BATCHES} weeks: "
    f"{classical_result_no_drift.running_means[-1]:.1%} / {classical_result_no_drift.confidence_bounds[-1]:.1%}"
)

In [ ]:
plot_monitor_vs_baseline(
    lower_no_drift,
    blind_lower_no_drift,
    pprm_result_no_drift,
    classical_result_no_drift,
    naive_alarms_no_drift,
    blind_naive_alarms_no_drift,
    title="Scenario 1: stationary stream (no drift)",
)

At some point, ordinary sampling noise pushes one of the confidence intervals above the threshold, and the naive baselines raise false alarms, purely from **peeking**: with a fresh, independent-looking test every week, the chance of at least one false alarm climbs the longer you keep checking, even though the hallucination rate did not move. The cluster-blind baseline is the more exposed of the two: its intervals are visibly narrower than the cluster-aware ones, not because it knows more, but because it counts every turn as new, independent evidence while turns of the same conversation largely repeat each other. Its overconfident intervals cross the threshold on noise more easily. `AsymptoticClusteredPPRM` and `AsymptoticClassicalRM`, in contrast, never alarm across the full stream: their running means stay below the threshold, with anytime-valid lower bounds safely underneath it as well.

## Scenario 2: slow drift, early alarm

Now we generate data with a progressive drift: partway through the stream, the hallucination rate ramps up to an unacceptably high level.

In [ ]:
DRIFT_START = 15
DRIFT_AMPLITUDE = 0.35
RAMP_LENGTH = 30

true_means_slow_drift = np.hstack(
    [
        np.full(DRIFT_START, STATIONARY_MEAN),
        np.linspace(STATIONARY_MEAN, STATIONARY_MEAN + DRIFT_AMPLITUDE, RAMP_LENGTH),
        np.full(N_BATCHES - RAMP_LENGTH - DRIFT_START, STATIONARY_MEAN + DRIFT_AMPLITUDE),
    ]
)
y_true_slow_drift, y_proxy_slow_drift, clusters_slow_drift, batches_slow_drift = simulate_batches(
    true_means_slow_drift, random_seed=5
)

lower_slow_drift, _, naive_alarms_slow_drift = naive_per_batch_alarms(
    y_true_slow_drift,
    y_proxy_slow_drift,
    clusters_slow_drift,
    batches_slow_drift,
    THRESHOLD,
    N_BATCHES,
    cluster_aware=True,
)
blind_lower_slow_drift, _, blind_naive_alarms_slow_drift = naive_per_batch_alarms(
    y_true_slow_drift,
    y_proxy_slow_drift,
    clusters_slow_drift,
    batches_slow_drift,
    THRESHOLD,
    N_BATCHES,
    cluster_aware=False,
)
pprm_result_slow_drift = AsymptoticClusteredPPRM().detect(
    y_true_slow_drift,
    y_proxy_slow_drift,
    clusters_slow_drift,
    batches_slow_drift,
    higher_is_better=False,
    threshold=THRESHOLD,
    confidence_level=CONFIDENCE_LEVEL,
    metric_name="Hallucination Rate",
)
classical_result_slow_drift = AsymptoticClassicalRM().detect(
    y_true_slow_drift,
    batches_slow_drift,
    higher_is_better=False,
    threshold=THRESHOLD,
    confidence_level=CONFIDENCE_LEVEL,
    metric_name="Hallucination Rate",
)

for name, alarms in [
    ("Naive baseline (cluster-aware)", naive_alarms_slow_drift),
    ("Naive baseline (cluster-blind)", blind_naive_alarms_slow_drift),
]:
    alarm_weeks = np.flatnonzero(alarms) + 1
    print(f"{name} alarms: {len(alarm_weeks)} weeks, first at week {alarm_weeks[0]}")
pprm_alarm_index = pprm_result_slow_drift.first_alarm_index
if pprm_alarm_index is not None:
    print(f"Clustered PPRM drift detected: True, first alarm at week {pprm_alarm_index + 1}")
classical_alarm_index = classical_result_slow_drift.first_alarm_index
if classical_alarm_index is not None:
    print(f"Classical monitor drift detected: True, first alarm at week {classical_alarm_index + 1}")

In [ ]:
plot_monitor_vs_baseline(
    lower_slow_drift,
    blind_lower_slow_drift,
    pprm_result_slow_drift,
    classical_result_slow_drift,
    naive_alarms_slow_drift,
    blind_naive_alarms_slow_drift,
    title="Scenario 2: drift occurrence during deployment",
)

Observe that the peeking baselines alarm earlier. However, this occurs even before the drift happens and, therefore, qualifies as a false alarm. In real time, a team watching either naive baseline has no way to tell a validated alarm from a lucky (or unlucky) one, because its error rate was never controlled for repeated testing in the first place: Scenario 1 already showed it firing on a completely stationary stream. Accounting for the conversation structure is not enough on its own to fix this, since the cluster-aware baseline still peeks. On the other hand, the anytime-valid alarms come fairly early after a consistent drift can be observed and not before, which makes them more trustworthy. Between the two monitors, `AsymptoticClusteredPPRM` alarms sooner than `AsymptoticClassicalRM` in this run: the correlated proxy labels sharpen its per-batch estimate on top of the human labels both monitors share, shrinking its confidence sequence and letting it accumulate evidence of the drift faster.

`AsymptoticClusteredPPRM`'s alarm carries no ambiguity: it is backed by the same anytime-valid guarantee demonstrated in Scenario 1, so under no drift the chance it would ever fire at all across the whole monitoring horizon corresponds to the used confidence level, and that guarantee holds with the conversation structure taken into account. `AsymptoticClassicalRM` serves here as the label-only reference: it treats every turn as an independent sample, so its guarantee is only as good as that assumption. By the end of the stream, both running means and their bounds are clearly above the threshold.

## Reading the result object

In both scenarios, `AsymptoticClusteredPPRM` returned a `PredictionPoweredMeanMonitoringResult`. Its public surface covers everything used above, plus a formatted summary.

In [ ]:
print("Clustered PPRM")
print(f"{'Drift detected':<32}: {pprm_result_slow_drift.drift_detected}")
print(f"{'First alarm index':<32}: {pprm_result_slow_drift.first_alarm_index}")
print(f"{'Alarm threshold':<32}: {pprm_result_slow_drift.alarm_threshold}")
print(f"{'Confidence level':<32}: {pprm_result_slow_drift.confidence_level}")
print(f"{'Running means (last 3)':<32}: {np.round(pprm_result_slow_drift.running_means[-3:], 3)}")
print(f"{'Confidence bounds (last 3)':<32}: {np.round(pprm_result_slow_drift.confidence_bounds[-3:], 3)}")
print(f"{'True labels per batch (last 3)':<32}: {pprm_result_slow_drift.batch_n_true[-3:]}")
print(f"{'Proxy labels per batch (last 3)':<32}: {pprm_result_slow_drift.batch_n_proxy[-3:]}")

`batch_n_true` and `batch_n_proxy` count a batch's labels at the turn level, not at the conversation level: `batch_n_proxy` is the number of turns judged in each batch, unaffected by how many conversations the batch contains, exactly as it would be for an unclustered stream. `batch_n_true` is the number of turns belonging to the human-reviewed conversations, so it fluctuates from one batch to the next with the length of the sampled conversations.

## Summary: accounting for conversations keeps anytime-valid monitoring honest

| | Per-batch baseline (cluster-blind) | `AsymptoticClassicalRM` | `AsymptoticClusteredPPRM` |
|-|---------------------|-------------------|-------------------|
| Valid under repeated looks? | ❌ No | ✅ Yes, for independent samples | ✅ Yes |
| Accounts for within-conversation correlation? | ❌ No | ❌ No, treats every turn as independent | ✅ Yes |
| Uses proxy labels? | ✅ Yes | ❌ No | ✅ Yes |
| Detects a sustained drift? | 🟠 Yes, but prone to early false alarms | ✅ Yes | ✅ Yes, no later than the classical monitor |

**Key takeaways:**

1. **Repeated significance testing is not a monitor.** Scenario 1 showed the per-batch baselines raise a false alarm on a perfectly stationary stream, purely from checking every week.
2. **Ignoring the conversation structure gives overconfident bounds.** Turns of the same conversation are correlated, so counting each turn as independent evidence yields intervals that are too narrow. Accounting for the clusters widens them to what the data actually supports, and that is what keeps alarms from firing on noise.
3. **An anytime-valid confidence sequence makes checking every week safe.** `AsymptoticClusteredPPRM`'s guarantee holds simultaneously across all looks, so its alarm in Scenario 2 can be trusted the moment it fires, without waiting to see whether later weeks confirm it.
4. **Validity is the point, not tightness.** Unlike the stratified and multi-proxy settings, where more information tightens the confidence sequence, the cluster-aware monitor is not narrower than a naive one. It is the one whose bounds mean what they claim, while still using the proxy labels to alarm no later than the label-only monitor.

---

*Want to go further? The [Monitors user guide](../../../user_guide/monitors/) derives the anytime-valid confidence sequence used here from first principles, including the variance formula that accounts for clustered samples.*